### Import of required Libraries

In [31]:
# --- Standard libraries ---
import json
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- IPython (for Jupyter display utilities) ---
from IPython.display import SVG, clear_output

# --- Plotly (interactive plotting) ---
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots  # optional, falls benötigt

# --- Scientific / statistical tools ---
from scipy import stats

# --- Scikit-learn: preprocessing, model selection, metrics, calibration ---
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder

# --- LightGBM ---
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from optuna.integration import lightgbm as lgb_optuna

# --- TensorFlow / Keras (for AROT model) ---
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.layers import Concatenate, Dense, Dropout, Flatten

# --- Domain-specific library: traffic data ---
from traffic.core import Traffic


In [32]:
pd.set_option("display.max_columns", None)

#### Import Data

In [33]:
final_table = pd.read_csv('Final_Table_V2.csv')

#### Create a Copy of Final Table to work with

In [34]:
data = final_table.copy()

##### Decrease the number of Airlines and replace those with not enough appearances with unknown + parking position

In [35]:
def replace_rare_icaos(group, threshold=50):
    # Count ICAO codes within the group
    counts = group['ICAO Code'].value_counts()
    rare_icaos = counts[counts < threshold].index
    # Replace rare ICAOs with e.g., "Rare_North" or "Rare_South"
    group['ICAO Code'] = group['ICAO Code'].apply(
        lambda x: f"Rare_{group.name}" if x in rare_icaos else x
    )
    return group

# Apply the function group-wise by Gate Region
data = data.groupby('Gate Region').apply(replace_rare_icaos).reset_index(drop=True)


C:\Users\hanv\AppData\Local\Temp\ipykernel_6792\921608701.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('Gate Region').apply(replace_rare_icaos).reset_index(drop=True)


##### Decrease the number of A/C Types and replace those with not enough appearances with unknown + parking position

In [36]:
def replace_rare_ac_types(group, threshold=50):
    # Count A/C Types within the group
    counts = group['A/C Type'].value_counts()
    rare_types = counts[counts < threshold].index
    # Replace rare A/C Types with e.g., "Rare_Heavy", "Rare_Light", etc.
    group['A/C Type'] = group['A/C Type'].apply(
        lambda x: f"Rare_{group.name}" if x in rare_types else x
    )
    return group

# Apply the function group-wise by ICAO Weight Turbulence Category
data = data.groupby('ICAO Weight Turbulence Category').apply(replace_rare_ac_types).reset_index(drop=True)


C:\Users\hanv\AppData\Local\Temp\ipykernel_6792\2761709303.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('ICAO Weight Turbulence Category').apply(replace_rare_ac_types).reset_index(drop=True)


#### Determine the Difference of the Entry Time between successive Arrivals

In [37]:
# Cleanly sort and reset integer index
data = data.sort_values(by="Time of Prediction", ignore_index=True)

# Parse datetime fields
data['Entry Time'] = pd.to_datetime(data['Entry Time'])
data['Exit Time']  = pd.to_datetime(data['Exit Time'])

# Time difference to the next entry time (in seconds)
data['Entry Time Difference (s)'] = (
    data['Entry Time'].shift(-1) - data['Entry Time']
).dt.total_seconds()


#### Determine Time Buffers

In [38]:
# Cleanly sort and reset integer index
data = data.sort_values(by="Time of Prediction", ignore_index=True)

# Time difference to the next entry time (in seconds)
data['Time Difference (s)'] = (
    data['Entry Time'].shift(-1) - data['Exit Time']
).dt.total_seconds()


In [39]:
data

,Flight ID,Entry Time,Exit Time,ICAO Code,A/C Type,ICAO Aircraft Type,Propulsion Type,Number of Engines,MALW [kg],ICAO Weight Turbulence Category,Gate Region,RET,ROT [s],Time of Prediction,Wind speed [kt],Wind direction [°],Visibility Category,Temperature [°C],Meteo idx,Precipitation [mm],Precipitation idx,Precipitation Timestamp,Hour sin,Hour cos,Minute sin,Minute cos,Day of week sin,Day of week cos,Month sin,Month cos,Year sin,Year cos,Day/Night,No Wind,Wind Variable,Wind direction sin,Wind direction cos,Headwind [kt],Crosswind [kt],Traffic Intensity RWY 10,Traffic Intensity RWY 14,Departure Traffic Intensity RWY 16,Arrival Traffic Intensity RWY 16,Traffic Intensity RWY 16,Departure Traffic Intensity RWY 28,Arrival Traffic Intensity RWY 28,Traffic Intensity RWY 28,Traffic Intensity RWY 32,Departure Traffic Intensity RWY 34,Arrival Traffic Intensity RWY 34,Traffic Intensity RWY 34,Airport Departure Traffic Intensity,Airport Arrival Traffic Intensity,Total Airport Traffic Intensity,Time Reserve [s],Number of preceding Traffic,Preceding Traffic ICAO Code,Preceding Traffic A/C Type,Preceding Traffic ICAO Aircraft Type,Preceding Traffic Propulsion Type,Preceding Traffic Number of Engines,Preceding Traffic MALW [kg],Preceding Traffic ICAO Weight Turbulence Category,Preceding Traffic Gate,Preceding Aircraft Geoaltitude,Preceding Aircraft Speed,Distance to Preceding Aircraft,Number of Successive Traffic,Successive Traffic ID,Successive Traffic A/C Type,Successive Traffic Geoaltitude,Successive Traffic Groundspeed,Successive Traffic Track,Successive Traffic Vertical Rate,Successive Traffic Distance,Successive Traffic Track sin,Successive Traffic Track cos,Successor Aligned,ROT Previous Flight [s],Average ROT Previous 5 Flight [s],Average ROT Previous Flight (Intensity) [s],ROT Previous Flight same A/C Type [s],Average ROT Previous 5 Flights same A/C Type [s],Average ROT Previous Flight same A/C Type (Intensity) [s],Average ROT Previous Flight same A/C Type Total [s],ROT Previous Flight same ICAO Weight Category [s],Average ROT Previous 5 Flights same ICAO Weight Category [s],Average ROT Previous Flight same ICAO Weight Category (Intensity) [s],Average ROT Previous Flight same ICAO Weight Category Total [s],Average ROT Total [s],groundspeed at prediction,geoaltitude at prediction,vertical_rate at prediction,distance at prediction,station,reference_timestamp,temp_2m,temp_5cm,temp_surface,temp_chill,rh_2m,dewpoint_2m,vapour_pressure,pressure_qfe,pressure_qnh,pressure_qff,geopot_850,geopot_700,gust_1s_ms,wind_vec_ms,wind_scalar_ms,wind_dir,foehn_idx,wind_scalar_kmh,gust_3s_ms,gust_1s_kmh,gust_3s_kmh,precip_10min,snow_depth,rad_global,rad_diffuse,rad_lw_in,rad_lw_out,rad_sw_reflect,sunshine_10min,new_wind_speed_kt,new_headwind_kt,new_crosswind_kt,H1,H2,H3,Entry Time Difference (s),Time Difference (s)
0,SIA346_291674,2024-03-01 06:23:00+00:00,2024-03-01 06:23:57+00:00,SIA,B77W,L2J,Jet,2.0,251290.0,Heavy,North,H1,57.0,2024-03-01 06:21:26+00:00,1.0,0.0,3,5.0,16.0,0.6,30.0,2024-03-01 06:00:00+00:00,1.0,0.5,0.904508,0.206107,0.283058,0.049516,1.0,0.5,0.999013,0.531395,Day,False,True,0.000000,0.00000,1.00,1.00,0,0,0,0,0,7,0,7,0,0,0,0,7,0,7,NaN,0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,1325.000000,0.0,13.752771,0.0,NaN,NaN,16000.0,417.0,317.231857,0.0,19.097551,0.000000,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,163.0,2950.0,-896.0,4.003528,KLO,2024-03-01 06:20:00+00:00,5.1,4.4,4.6,5.1,97.1,4.7,8.5,959.0,1009.2,1010.3,NaN,NaN,1.5,1.1,1.1,106.0,NaN,4.0,1.5,5.4,5.4,0.4,0.0,2,1.0,336,NaN,NaN,0,2.159827,1.85,-1.12,0,0,0,582.0,525.0
1,ETH736_1162,2024-03-01 06:32:42+00:00,2024-03-01 06:33:36+00:00,ETH,A359,L2J,Jet,2.0,207000.0,Heavy,South,H1,54.0,2024-03-01 06:31:12+00:00,1.0,0.0,3,5.0,16.0,0.6,30.0,2024-03-01 06:00:00+00:00,1.0,0.5,0.447736,0.002739,0.283058,0.049516,1.0,0.5,0.999013,0.531395,Day,False,True,0.000000,0.00000,1.00,1.00,0,1,0,0,0,9,0,9,0,0,0,0,9,1,10,525.0,0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,1325.000000,0.0,13.752771,0.0,NaN,

#### Extract only Data which vacated the Runway via an valid Exit Taxiway

In [40]:
data = data[(data['RET'] == 'H1') | (data['RET'] == 'H2') | (data['RET'] == 'H3')] # H3 included for Time Buffer Plot. Will be removed in another step

#### Define a lower threshold of 39 secondes to remove faulty data

In [41]:
data = data[data['ROT [s]'] >=39]

In [42]:
data = data[data['Entry Time Difference (s)'] > 0]

#### Set Categorical Types

In [43]:
data['A/C Type'] = data['A/C Type'].astype('category')
data['ICAO Code'] = data['ICAO Code'].astype('category')
data['ICAO Weight Turbulence Category'] = data['ICAO Weight Turbulence Category'].astype('category')
data['Propulsion Type'] = data['Propulsion Type'].astype('category')
data['Day/Night'] = data['Day/Night'].astype('category')
data['No Wind'] = data['No Wind'].astype('category')
data['Wind Variable'] = data['Wind Variable'].astype('category')
data['Gate Region'] = data['Gate Region'].astype('category')

In [44]:
# List of all flight IDs to be removed
invalid_ids = [
    # H1 - Long ROTs
    "SAS841_74600", "AEA91MP_5645", "SWR4GV_272636", "EDW675Z_191415",
    "SWR504W_144961", "SPEMC_64994", "SWR9GW_88322", "SWR2EP_134507",
    "SWR296W_280795", "SWR1TC_125514", "SWR969_80375", "DLH8WN_29304",
    # H1 - Short ROTs
    "SWR4CQ_236343", "DISAG_33952", "SWR9LZ_128323", "EWG95AW_269167",
    "SWR6MW_223600", "SWR3BY_105353", "SWR339H_132594",
    # H1 - Other observations
    "AUA55C_45096", "SXS7SL_251961", "SKV444_46316", "EJU5189_45326",
    "LDX20C_46560", "EWG59L_261077", "VJT518_261031", "EJU96HY_45377",
    "OEFXY_45632", "OEFXY_45636",
    # H2 - Long ROTs
    "SWR589X_197642", "KAL917_289693", "SWR262C_114944", "AUA5U_46704",
    "SWR2HE_111023", "SWR2DG_96864", "DCSEB_32456", "EWG64R_20879",
    "SWR71Y_279675", "SWR457_141457", "SWR539L_276851", "SWR4MY_85273",
    "SWR438A_87128", "SWR5KP_133206",
    # H2 - Short ROTs
    "SWR4GV_201277", "DLH8CM_16669", "SXS9UC_253202", "SWR30M_194526",
    "SWR63D_99758", "SWR296W_285509", "SWR5GZ_115709", "EDW215P_195137",
    "SWR2DG_196962", "SWR2121_121074", "SWR98X_84057", "SWR8WK_160440",
    "SWR5CL_121762", "SWR189_202667", "SWR9LZ_126613", "SWR622P_79915",
    # H3 - Short ROTs
    "LMJ920F_67939",
    # H3 - Other
    "RGA01_245094",
]

# Filter out invalid flight IDs
data = data[~data["Flight ID"].isin(invalid_ids)].copy()

# Filter out unwanted aircraft types
invalid_ac_types = ["SF25", "PA18", "PA31", "PA34", "TB20", "C207", "DA40"]
data = data[~data['A/C Type'].isin(invalid_ac_types)]

# Create a copy for plotting
data_buffer_plot = data.copy()


##### Remove H3 from dataset

In [45]:
data = data[(data['RET'] == 'H1') | (data['RET'] == 'H2')]

#### Remove Flights which are considered as irrelevant due to their Time Difference

In [46]:
heavy = data['ICAO Weight Turbulence Category'].isin(['Heavy', 'Super', 'H', 'J'])

mask = (heavy  & (data['Entry Time Difference (s)'] <= 180)) | \
       (~heavy & (data['Entry Time Difference (s)'] <= 120))

data = data[mask]

#### Define X_ret for RET prediction

In [47]:
X_ret = data[[
    'A/C Type',
    'ICAO Code',
    'MALW [kg]',
    'Gate Region',
    'new_headwind_kt',
    'new_crosswind_kt',
    'Visibility Category',
    'temp_2m',
    'pressure_qnh',
    'precip_10min',
    'Month sin',
    'Month cos',
    'Day of week sin',
    'Day of week cos',
    'groundspeed at prediction',
]]

#### Filter categorical Columns in X_ret

In [48]:
categorical_columns = [
    'ICAO Code',
    'ICAO Weight Turbulence Category',
    'Gate Region',
]


#### Identify Categorical Feature Indices

In [49]:
categorical_columns_in_X = [col for col in categorical_columns if col in X_ret.columns]
categorical_feature_indices = [X_ret.columns.get_loc(col) for col in categorical_columns_in_X]

#### Remove NaN from the Dataset (only based on X_ret)

In [50]:
X_ret = X_ret.dropna()

#### Remove special characters from the column names

In [51]:
X_ret_columns_names = X_ret.columns
X_ret.columns = X_ret.columns.str.replace(r"[\[\]<>]", "", regex=True)

#### Define Target

In [52]:
y_ret = data.loc[X_ret.index, ['RET']]

#### Encode Target

In [53]:
# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y_ret)

c:\Users\hanv\Desktop\Master\EVA\Code\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


#### Split Set in a Set for RET Predcition Model and in a Set for ROT Prediction Model (ROT Set is the Test Set of the RET Set)

In [54]:
X_ret, X_rot, y_ret, y_rot = train_test_split(
    X_ret, y_encoded, test_size=0.5, random_state=42
)

#### Define Train and Validation Set of the RET Set (Validation Set will no be used)

In [55]:
X_ret_train, X_ret_val, y_ret_train, y_ret_val = train_test_split(
    X_ret, y_ret, test_size=0.2, random_state=42
)

In [56]:
X_ret_train

,A/C Type,ICAO Code,MALW kg,Gate Region,new_headwind_kt,new_crosswind_kt,Visibility Category,temp_2m,pressure_qnh,precip_10min,Month sin,Month cos,Day of week sin,Day of week cos,groundspeed at prediction
73098,B38M,LOT,69309.0,South,-2.11,2.53,9,8.3,999.2,0.0,0.750000,0.933013,0.500000,1.000000,168.0
61960,B77W,SWR,251290.0,North,-5.25,3.31,10,8.7,1016.0,0.0,0.250000,0.933013,0.109084,0.811745,166.0
37047,A321,SWR,77800.0,South,-1.01,3.36,9,28.6,1016.4,0.0,0.066987,0.250000,0.500000,1.000000,164.0
13230,BCS3,SWR,60600.0,East,0.86,6.75,8,12.9,1012.1,0.0,0.750000,0.066987,0.500000,1.000000,163.0
26053,B77W,SWR,251290.0,North,-2.71,4.73,10,26.2,1018.3,0.0,0.500000,0.000000,0.283058,0.049516,177.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50335,BCS3,SWR,60600.0,South,-0.37,0.46,10,10.3,1011.6,0.0,0.066987,0.750000,0.987464,0.388740,150.0
47799,A320,SWR,66000.0,South,1.18,1.26,10,21.8,1015.2,0.0,0.000000,0.500000,0.109084,0.811745,168.0
15310,BCS1,SWR,54200.0,South,0.65,-7.75,10,24.9,1004.6,0.0,0.750000,0.066987,0.890916,0.811745,151.0
13420,A343,EDW,192000.0,South,-1.83,1.15,7,11.7,1016.2,0.0,0.750000,0.066987,0.890916,0.811745,161.0


### Train Naive Baseline Model and evaluate on AROT Data Set (Test Set)

In [57]:
# Create training DataFrame with aircraft type and corresponding RET label
train_df = pd.DataFrame({
    "A/C Type": X_ret_train["A/C Type"].astype(str).values,
    "RET": pd.Series(y_ret_train).astype(str).values
})

# Determine the global most frequent RET (fallback for unknown aircraft types)
global_ret = train_df["RET"].value_counts().idxmax()

# Create mapping: Aircraft Type -> most frequent RET in training data
mapping = (
    train_df.groupby("A/C Type")["RET"]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

# Predict RET for ROT dataset using mapping, fallback to global RET if unseen
y_pred_rot = (
    X_rot["A/C Type"]
    .astype(str)
    .map(mapping)
    .fillna(global_ret)
)

# Convert true labels to string format for comparison
y_true = pd.Series(y_rot).astype(str)

# Evaluate overall accuracy
print("Accuracy:", accuracy_score(y_true, y_pred_rot))

# Print detailed classification metrics (precision, recall, F1-score)
print("\nClassification Report:")
print(classification_report(y_true, y_pred_rot))

Accuracy: 0.7777697893450284

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.98      0.86     19649
           1       0.89      0.28      0.42      8169

    accuracy                           0.78     27818
   macro avg       0.83      0.63      0.64     27818
weighted avg       0.80      0.78      0.73     27818

